# 3-Agent Backtesting Workflow with Responses API

This notebook implements a comprehensive backtesting system with 3 specialized agents using OpenAI Responses API:

1. **Technical Analysis Agent**: Analyzes market data and generates trading signals
2. **Simulator Agent**: Simulates trades based on TA analysis, outputs TradeRecord model with structured output
3. **Reviewer Agent**: Reviews simulation results with code interpreter tool

## Architecture

```
Date Range Input
       ↓
Sliding Windows (7-day analysis windows)
       ↓
┌──────────────────────────────────────┐
│  Phase 1: Technical Analysis         │
│  - Analyze market data               │
│  - Generate trading signals          │
│  - Identify entry/exit points        │
└──────────────────────────────────────┘
       ↓
┌──────────────────────────────────────┐
│  Phase 2: Trade Simulation           │
│  - Execute trades based on TA        │
│  - Track positions                   │
│  - Output TradeRecord models         │
│  - Uses text_format for structured   │
└──────────────────────────────────────┘
       ↓
┌──────────────────────────────────────┐
│  Phase 3: Review & Analysis          │
│  - Code interpreter evaluation       │
│  - Performance metrics               │
│  - Strategy improvement suggestions  │
└──────────────────────────────────────┘
```

## Setup and Imports

In [2]:
import os
import json
import time
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Any
from pathlib import Path

import pandas as pd
import mysql.connector
from pydantic import BaseModel, Field
from openai import OpenAI
import pytz
from dotenv import load_dotenv

load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


## Pydantic Models

In [3]:
class TradeRecord(BaseModel):
    """Complete trade record output from Simulator Agent"""
    
    id: int = Field(..., description="Số thứ tự giao dịch")
    entry_time: datetime = Field(..., description="Thời gian vào lệnh (UTC+7)")
    exit_time: datetime = Field(..., description="Thời gian ra lệnh (UTC+7)")
    order_type: str = Field(..., description="Loại lệnh: Long hoặc Short")

    # --- Dữ liệu giá & kết quả ---
    entry_price: float = Field(..., description="Giá vào lệnh theo TA")
    exit_price: float = Field(..., description="Giá ra thực tế (market)")
    target_price: Optional[float] = Field(None, description="Giá TP theo TA (nếu có)")
    stop_price: Optional[float] = Field(None, description="Giá SL theo TA (nếu có)")
    pnl_expected: Optional[float] = Field(None, description="Lợi nhuận dự kiến theo TA (%)")
    pnl_real: Optional[float] = Field(None, description="Lợi nhuận/lỗ thực tế (%)")
    deviation: Optional[float] = Field(None, description="Sai lệch giữa thực tế và dự kiến (%)")

    # --- Thời gian & kỹ thuật ---
    holding_time_hours: Optional[float] = Field(None, description="Thời gian giữ lệnh (giờ)")
    technical_reason: Optional[str] = Field(None, description="Lý do kỹ thuật mở lệnh")
    ta_reference: Optional[str] = Field(None, description="Dẫn chứng từ báo cáo kỹ thuật")

    # --- Đánh giá kết quả ---
    result: Optional[str] = Field(None, description="Kết quả: Lãi / Lỗ")
    ta_assessment: Optional[str] = Field(None, description="Nhận định theo phân tích kỹ thuật")
    market_result: Optional[str] = Field(None, description="Kết quả thực tế thị trường")
    deviation_reason: Optional[str] = Field(None, description="Nguyên nhân chênh lệch giữa TA và thực tế")
    improvement_note: Optional[str] = Field(None, description="Gợi ý cải thiện chiến lược")

    # --- Thống kê mở rộng (tùy chọn) ---
    volume_condition: Optional[str] = Field(None, description="Tình trạng volume tại thời điểm giao dịch")
    oi_change_24h: Optional[float] = Field(None, description="Mức thay đổi OI (%) trong 24h")
    funding_rate: Optional[float] = Field(None, description="Funding rate tại thời điểm vào lệnh")
    atr_value: Optional[float] = Field(None, description="ATR tại thời điểm vào lệnh (nếu dùng quản trị rủi ro ATR)")


class TradeSimulationOutput(BaseModel):
    """Wrapper model for Simulator output - contains list of trades"""
    trades: List[TradeRecord] = Field(..., description="List of simulated trades")


class TAReviewOutput(BaseModel):
    """Structured output for TA Review Agent - evaluates TA prediction accuracy"""
    
    # Overall Accuracy Metrics
    accuracy_score: float = Field(..., description="Overall accuracy score from 0-10 for TA predictions")
    prediction_alignment: str = Field(..., description="How well predictions aligned with actual outcomes: Excellent/Good/Fair/Poor")
    
    # Price Movement Analysis
    price_direction_correct: bool = Field(..., description="Did TA correctly predict price direction?")
    price_target_accuracy: Optional[float] = Field(None, description="Percentage accuracy of price targets (if applicable)")
    support_resistance_valid: bool = Field(..., description="Were support/resistance levels accurate?")
    
    # Technical Indicators Analysis  
    volume_analysis_correct: bool = Field(..., description="Was volume analysis accurate?")
    oi_funding_analysis_correct: bool = Field(..., description="Were OI and funding rate analyses accurate?")
    
    # Detailed Commentary (Vietnamese)
    strengths: str = Field(..., description="What the TA analysis got RIGHT (2-3 points in Vietnamese)")
    weaknesses: str = Field(..., description="What the TA analysis got WRONG or missed (2-3 points in Vietnamese)")
    
    # Market Context Analysis
    market_conditions: str = Field(..., description="Actual market conditions during the period (Vietnamese)")
    unexpected_events: Optional[str] = Field(None, description="Any unexpected market events that affected accuracy (Vietnamese)")
    
    # Improvement Recommendations
    ta_improvements: str = Field(..., description="Specific suggestions to improve TA methodology (Vietnamese)")
    indicator_recommendations: Optional[str] = Field(None, description="Additional indicators that should have been used (Vietnamese)")
    
    # Summary
    overall_assessment: str = Field(..., description="2-3 paragraph overall assessment in Vietnamese")


class BacktestWindow(BaseModel):
    """Represents a single backtesting window"""
    window_id: int
    start_date: datetime
    end_date: datetime
    analysis_data: Optional[str] = None  # TA predictions
    trades: Optional[List[TradeRecord]] = None  # Simulator result
    review: Optional[str] = None  # TA Review result
    actual_market_data: Optional[str] = None  # Actual market outcomes for review


print("✓ Pydantic models defined")

✓ Pydantic models defined


## Database Connection

In [4]:
def get_db_connection():
    """
    Create MySQL database connection with timeout

    Optimizations:
    - 5 second connection timeout to prevent hanging
    - Explicit connection parameters for better control
    """
    return mysql.connector.connect(
        host=os.getenv("MYSQL_HOST", "localhost"),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD", ""),
        database=os.getenv("MYSQL_DATABASE", "crypto_data"),
        connection_timeout=5
    )


def fetch_market_data(
    symbol: str,
    start_time: datetime,
    end_time: datetime
) -> Dict[str, Any]:
    """
    Fetch comprehensive market data for Technical Analysis Agent
    
    Includes:
    - 4h klines (OHLCV)
    - Funding rates (compressed with statistics)
    - Open Interest (4h aggregated)
    
    Args:
        symbol: Trading pair (e.g., 'BTCUSDT')
        start_time: Start datetime (inclusive)
        end_time: End datetime (exclusive)
    
    Returns:
        Dict with kline, funding, and open_interest DataFrames
    """
    conn = get_db_connection()
    cursor = conn.cursor(dictionary=True)
    
    try:
        symbol = symbol.lower().strip()
        data = {}
        
        # --- 4h Klines (OHLCV only) ---
        # Use close_time to get exact candles within window [start_time, end_time)
        cursor.execute("""
            SELECT close_time, open_price, high_price, low_price, close_price, volume
            FROM finance_services.crypto_kline_hours
            WHERE `interval`='4h' AND symbol=%s
            AND close_time >= %s
            AND close_time < %s
            ORDER BY close_time
        """, (symbol, start_time, end_time))
        
        kline_rows = cursor.fetchall()
        if kline_rows:
            df_kline = pd.DataFrame(kline_rows)
            df_kline['close_time'] = pd.to_datetime(df_kline['close_time'])
            # Filter zero volume and round
            df_kline = df_kline[df_kline['volume'] > 0].copy()
            float_cols = df_kline.select_dtypes(include='float').columns
            df_kline[float_cols] = df_kline[float_cols].round(3)
            data['kline'] = df_kline
        else:
            data['kline'] = pd.DataFrame()
        
        # --- Funding Rates (compressed) ---
        cursor.execute("""
            SELECT funding_time, funding_rate
            FROM finance_services.futures_funding_rates
            WHERE symbol=%s
            AND funding_time >= %s
            AND funding_time < %s
            ORDER BY funding_time
        """, (symbol, start_time, end_time))
        
        funding_rows = cursor.fetchall()
        if funding_rows:
            df_fund = pd.DataFrame(funding_rows)
            df_fund = df_fund.sort_values("funding_time").reset_index(drop=True)
            df_fund["funding_delta"] = df_fund["funding_rate"].diff().fillna(0) * 1e4
            
            base_rate = float(df_fund["funding_rate"].iloc[0])
            deltas = df_fund["funding_delta"].iloc[1:]
            
            # Compressed funding data
            data['funding'] = {
                "base_rate": base_rate,
                "stats": {
                    "avg_delta": round(float(deltas.mean()), 3),
                    "max_delta": round(float(deltas.max()), 3),
                    "min_delta": round(float(deltas.min()), 3),
                    "volatility": round(float(deltas.std()), 3)
                },
                "series": df_fund.iloc[::max(1, len(df_fund)//8)][["funding_time", "funding_rate"]]
                          .rename(columns={"funding_time": "ts", "funding_rate": "rate"})
                          .round(8)
                          .to_dict(orient="records"),
                "count": len(df_fund)
            }
        else:
            data['funding'] = {}
        
        # --- Open Interest (4h aggregation) ---
        cursor.execute("""
            SELECT 
                DATE_FORMAT(MIN(timestamp),'%Y-%m-%d %H:00:00') AS ts,
                AVG(open_interest_usd) AS open_interest_usd,
                AVG(open_interest_coin) AS open_interest_coin
            FROM finance_services.futures_open_interests
            WHERE symbol=%s
            AND timestamp >= %s
            AND timestamp < %s
            GROUP BY FLOOR(UNIX_TIMESTAMP(timestamp)/(4*3600))
            ORDER BY ts
        """, (symbol, start_time, end_time))
        
        oi_rows = cursor.fetchall()
        if oi_rows:
            df_oi = pd.DataFrame(oi_rows)
            df_oi['ts'] = pd.to_datetime(df_oi['ts'])
            float_cols = df_oi.select_dtypes(include='float').columns
            df_oi[float_cols] = df_oi[float_cols].round(3)
            data['open_interest'] = df_oi
        else:
            data['open_interest'] = pd.DataFrame()
        
        return data
        
    finally:
        cursor.close()
        conn.close()


def fetch_simulator_data(
    symbol: str,
    start_time: datetime,
    end_time: datetime
) -> pd.DataFrame:
    """
    Fetch simplified market data for Simulator Agent
    
    Includes:
    - 1h klines (close price, volume, and time)
    
    Args:
        symbol: Trading pair (e.g., 'BTCUSDT')
        start_time: Start datetime (inclusive)
        end_time: End datetime (exclusive)
    
    Returns:
        DataFrame with columns: close_time, price, volume
    """
    conn = get_db_connection()
    cursor = conn.cursor(dictionary=True)
    
    try:
        symbol = symbol.lower().strip()
        
        cursor.execute("""
            SELECT 
                close_time, 
                close_price AS price,
                volume
            FROM finance_services.crypto_kline_hours
            WHERE `interval` = '1h'
              AND symbol = %s
              AND close_time >= %s
              AND close_time < %s
            ORDER BY close_time
        """, (symbol, start_time, end_time))
        
        rows = cursor.fetchall()
        if not rows:
            return pd.DataFrame(columns=["close_time", "price", "volume"])
        
        df = pd.DataFrame(rows)
        df['close_time'] = pd.to_datetime(df['close_time'])
        df['price'] = df['price'].astype(float).round(3)
        df['volume'] = df['volume'].astype(float).round(3)
        
        return df
        
    finally:
        cursor.close()
        conn.close()



# Test connection
try:
    conn = get_db_connection()
    conn.close()
    print("✓ Database connection successful")
except Exception as e:
    print(f"✗ Database connection failed: {e}")

✓ Database connection successful


## Sliding Window Generator

In [5]:
def generate_sliding_windows(
    start_date: datetime,
    end_date: datetime,
    window_size_days: int = 7,
    slide_days: int = 3
) -> List[BacktestWindow]:
    """
    Generate sliding windows for backtesting
    
    Args:
        start_date: Overall start date
        end_date: Overall end date
        window_size_days: Size of each window (default: 7 days)
        slide_days: Days to slide for next window (default: 3 days)
    
    Returns:
        List of BacktestWindow objects
    """
    windows = []
    window_id = 0
    
    current_start = start_date
    
    while current_start + timedelta(days=window_size_days) <= end_date:
        current_end = current_start + timedelta(days=window_size_days)
        
        windows.append(BacktestWindow(
            window_id=window_id,
            start_date=current_start,
            end_date=current_end
        ))
        
        window_id += 1
        current_start += timedelta(days=slide_days)
    
    return windows


# Test sliding window generation
test_start = datetime(2025, 5, 13, tzinfo=pytz.UTC)
test_end = datetime(2025, 10, 31, tzinfo=pytz.UTC)
test_windows = generate_sliding_windows(test_start, test_end)

print(f"✓ Generated {len(test_windows)} windows")
for window in test_windows:
    print(f"  Window {window.window_id}: {window.start_date} to {window.end_date}")


✓ Generated 55 windows
  Window 0: 2025-05-13 00:00:00+00:00 to 2025-05-20 00:00:00+00:00
  Window 1: 2025-05-16 00:00:00+00:00 to 2025-05-23 00:00:00+00:00
  Window 2: 2025-05-19 00:00:00+00:00 to 2025-05-26 00:00:00+00:00
  Window 3: 2025-05-22 00:00:00+00:00 to 2025-05-29 00:00:00+00:00
  Window 4: 2025-05-25 00:00:00+00:00 to 2025-06-01 00:00:00+00:00
  Window 5: 2025-05-28 00:00:00+00:00 to 2025-06-04 00:00:00+00:00
  Window 6: 2025-05-31 00:00:00+00:00 to 2025-06-07 00:00:00+00:00
  Window 7: 2025-06-03 00:00:00+00:00 to 2025-06-10 00:00:00+00:00
  Window 8: 2025-06-06 00:00:00+00:00 to 2025-06-13 00:00:00+00:00
  Window 9: 2025-06-09 00:00:00+00:00 to 2025-06-16 00:00:00+00:00
  Window 10: 2025-06-12 00:00:00+00:00 to 2025-06-19 00:00:00+00:00
  Window 11: 2025-06-15 00:00:00+00:00 to 2025-06-22 00:00:00+00:00
  Window 12: 2025-06-18 00:00:00+00:00 to 2025-06-25 00:00:00+00:00
  Window 13: 2025-06-21 00:00:00+00:00 to 2025-06-28 00:00:00+00:00
  Window 14: 2025-06-24 00:00:00+00

## Agent Prompts

These prompts are designed to remain 100% consistent across all batch requests.

In [6]:
# Technical Analysis Agent Prompt Template
TA_AGENT_PROMPT = """
Bạn là chuyên gia phân tích kỹ thuật crypto chuyên sâu, sử dụng logic phân tích đa khung thời gian 4H kết hợp dữ liệu thực tế gồm:
- Khung giờ mà bạn sẽ được cung cấp cũng như sử dụng là giờ Việt Nam, asia, UTC+7
- Giá hiện tại, giá mở cửa, biên độ dao động 24h.
- Các mức hỗ trợ, kháng cự.
- Volume, OI, Funding rate.
- Biểu đồ giá hoặc dữ liệu nến theo khung 4 giờ
## 1. Diễn biến giá (4H gần nhất)
* Giá hiện tại, giá mở cửa, biên độ 24h (% thay đổi).
* Mô tả hành động giá 4H gần nhất (mẫu nến, hướng xu hướng, độ dốc EMA, tín hiệu RSI/MACD nếu có).
* So sánh **Volume hiện tại với trung bình 20 kỳ** (VD: "Volume tăng 150% so với trung bình, xác nhận đà mua chủ động").
* **OI**: Tỷ lệ thay đổi trong 24h (%), phân tích tương quan với giá.

  * OI ↑ & giá ↑ → dòng tiền mới tham gia (tích cực)
  * OI ↑ & giá ↓/đi ngang → trap/ép long-short
* **Funding Rate**: Xu hướng (↑ / ↓ / trung lập), cảnh báo nếu lệch cân (VD: "Funding 0.08%, cao → rủi ro short squeeze").
## 2. Hỗ trợ – Kháng cự ngắn hạn
Liệt kê 2 mức hỗ trợ & 2 mức kháng cự rõ ràng (ghi giá cụ thể hoặc vùng biên ±0.5%).
→ Gợi ý ghi chú thêm "Hỗ trợ 1: ... | Kháng cự 1: ..."
## 3. Xác suất diễn biến (24–72 giờ tới)

| Kịch bản            | Xác suất (%) | Giải thích                                                                      |
| ------------------- | ------------ | ------------------------------------------------------------------------------- |
| Tăng / hồi kỹ thuật | XX%          | Giải thích xác suất hiện tại dựa vào những yếu tố định tính và định lượng nào? |
| Giảm / thủng hỗ trợ | YY%          | Giải thích xác suất hiện tại dựa vào những yếu tố định tính và định lượng nào? |

## 4. Gợi ý điểm vào – ra (Quản trị rủi ro định lượng)
**Long Setup:**
* Entry: vùng giá …
* Stop Loss: dựa theo **1.5x–2x ATR(14)** dưới hỗ trợ gần nhất
* Take Profit:

  * TP1: kháng cự gần nhất (chốt 50%)
  * TP2: kháng cự mạnh (chốt 50%)
* Khi TP1 đạt → **dời SL về điểm hòa vốn (Break-even)**
**Short Setup:**
* Entry: vùng giá …
* Stop Loss: **1.5x–2x ATR(14)** trên kháng cự gần nhất
* Take Profit tương tự (TP1/TP2 + quản trị dời SL)
## 5. Tổng hợp chiến lược

| Khung | Hỗ trợ | Kháng cự | Xu hướng | Kịch bản chính | Kịch bản phụ |
| ----- | ------ | -------- | -------- | -------------- | ------------ |
## 6. Gợi ý hành động (24–72h)

* **Trader ngắn hạn:** tập trung theo dõi vùng ... với điều kiện volume xác nhận hoặc OI tăng.
* Nếu Funding Rate cao → cảnh giác **trap tăng**; nếu thấp bất thường → đề phòng **short squeeze**.
* Chỉ vào lệnh khi có **xác nhận volume ≥120% trung bình 20 kỳ**.

Dữ liệu từ đồng {symbol}
Vào khoảng thời gian {start_date} đến {end_date}
Dữ liệu: {market_data}
"""


# Simulator Agent Prompt Template
SIMULATOR_AGENT_PROMPT = """
Bạn là chuyên gia mô phỏng giao dịch crypto (Backtesting Analyst).
Mục tiêu: đánh giá độ chính xác của bản phân tích kỹ thuật bằng cách giả lập các lệnh theo đúng chiến lược kỹ thuật, sau đó so sánh với dữ liệu thực tế để tính hiệu suất và sai lệch.
 Bạn sẽ được cung cấp:
Bản Technical Analysis (phân tích kỹ thuật):
Đây là cơ sở duy nhất để bạn ra quyết định vào/ra lệnh.
Bạn phải tuân thủ tuyệt đối, không được dựa vào dữ liệu thực tế khi đặt lệnh.

Dữ liệu thực tế (price history, volume, funding, OI...): Dùng chỉ để kiểm chứng kết quả, tính toán thời điểm chạm TP/SL, lãi/lỗ, và độ chính xác của phân tích kỹ thuật.
Quy tắc mô phỏng lệnh
Điểm vào/ra rõ ràng và có thời gian cụ thể (UTC+7)
Ví dụ:

Ngày 10/10/2025 – 10:00:00 vào lệnh Long ở 10000 USDT,
Chốt lời tại 11000 USDT (ngày 10/10/2025 – 18:00:00).

Số lượng lệnh
Có thể mở nhiều lệnh nếu bản phân tích kỹ thuật cho phép (ví dụ nhiều vùng hỗ trợ/kháng cự).
Mỗi lệnh phải được ghi lại riêng biệt với lý do kỹ thuật (RSI, MA, Breakout…).
Đánh giá định tính

Độ chính xác: …

Điều kiện phân tích kỹ thuật hoạt động tốt: (ví dụ: khi volume cao / thị trường có trend).

Sai lệch hoặc nguyên nhân thất bại: (funding lệch, volume yếu, biến động bất thường).

Kết luận: phân tích kỹ thuật đúng / sai / một phần đúng, kèm đề xuất cải thiện (thêm xác nhận RSI, ATR filter, khung 1D,…).

Nguyên tắc bắt buộc cho Agent

Không được "ăn gian" bằng cách nhìn dữ liệu thực để chọn điểm vào tốt hơn.

Không được sửa lệnh sau khi biết kết quả.

Mục tiêu không phải tối đa hóa lợi nhuận, mà là đánh giá độ tin cậy của phân tích kỹ thuật.

Here are the: Technical Analysis: {ta_analysis}
            Future Market Data: {future_market_data}
            Output Schema: {trade_record_schema}
"""


# NEW: TA Review Agent Prompt - Evaluates TA Prediction Accuracy
TA_REVIEWER_AGENT_PROMPT = """
Bạn là chuyên gia đánh giá phân tích kỹ thuật (TA Review Expert).

**MỤC TIÊU**: Đánh giá độ chính xác của bản phân tích kỹ thuật (TA) bằng cách so sánh DỰ ĐOÁN của TA với KẾT QUẢ THỰC TẾ của thị trường.

**QUAN TRỌNG**: 
- Đây là backtesting - bạn đã biết kết quả thực tế
- Nhiệm vụ của bạn là đánh giá TA có dự đoán ĐÚNG hay SAI
- KHÔNG đánh giá kết quả giao dịch, CHỈ đánh giá độ chính xác của phân tích kỹ thuật

## DỮ LIỆU BẠN SẼ NHẬN ĐƯỢC:

1. **Bản Phân Tích Kỹ Thuật (TA Predictions)**: 
   - Dự đoán xu hướng giá (tăng/giảm, xác suất)
   - Mức hỗ trợ/kháng cự dự kiến
   - Phân tích volume, OI, funding rate
   - Entry/Exit points được đề xuất
   - Kịch bản chính/phụ

2. **Dữ Liệu Thực Tế (Actual Market Outcomes)**:
   - Giá thực tế đã diễn ra như thế nào
   - Volume, OI, funding rate thực tế
   - Các mức hỗ trợ/kháng cự có hoạt động không

## TIÊU CHÍ ĐÁNH GIÁ:

### A. Độ Chính Xác Dự Đoán Giá (Price Prediction Accuracy)
- TA dự đoán giá tăng/giảm → thực tế có đúng không?
- Mức giá mục tiêu (TP) có được chạm không?
- Xác suất dự đoán (VD: 70% tăng) có hợp lý không?

### B. Hỗ Trợ - Kháng Cự (Support/Resistance Validity)
- Các mức hỗ trợ/kháng cự TA đưa ra có chính xác không?
- Giá có phản ứng tại các mức này không?
- TA có bỏ sót mức quan trọng nào không?

### C. Phân Tích Volume, OI, Funding (Indicator Analysis)
- Phân tích volume của TA có khớp với thực tế không?
- Dự đoán về OI và funding rate có chính xác không?
- TA có hiểu đúng ý nghĩa của các chỉ báo này không?

### D. Thời Gian & Kịch Bản (Timing & Scenarios)
- Khung thời gian dự đoán (24-72h) có chính xác không?
- Kịch bản chính/phụ của TA có xảy ra không?
- TA có bỏ sót kịch bản nào thực tế đã xảy ra không?

### E. Điều Kiện Thị Trường (Market Context)
- TA có nhận diện đúng điều kiện thị trường không? (trending/ranging/volatile)
- Có sự kiện bất ngờ nào TA không thể dự đoán không?

## YÊU CẦU ĐẦU RA:

Sử dụng code interpreter để:
1. Tính toán độ chính xác số lượng
2. Không xuất file mà mô tả báo cáo dưới dạng text
Đưa ra đánh giá bằng tiếng Việt bao gồm:
- **Điểm số tổng thể** (0-10)
- **Những gì TA phân tích ĐÚNG** (ưu điểm)
- **Những gì TA phân tích SAI** (nhược điểm)
- **Điều kiện thị trường thực tế**
- **Đề xuất cải thiện phương pháp TA**

---

### Bản Phân Tích Kỹ Thuật (TA Predictions):
{ta_analysis}

### Dữ Liệu Thị Trường Thực Tế (Actual Outcomes):
{actual_market_data}

### Dữ Liệu Giao Dịch Mô Phỏng (Reference):
{simulation_summary}
"""


print("✓ Agent prompts defined")

✓ Agent prompts defined


## Batch Request Helper Functions (Responses API Format)

In [7]:
import json
from typing import List, Dict
from pydantic import BaseModel
from dataclasses import asdict, is_dataclass
import numpy as np
def prepare_market_data_text(data: Dict[str, Any]) -> str:
    """
    Convert market data dict to formatted text for TA agent consumption
    
    Handles:
    - 4h klines (OHLCV)
    - Compressed funding rates with statistics
    - Open interest data
    
    Args:
        data: Dict from fetch_market_data() with kline, funding, open_interest
    
    Returns:
        Formatted text string
    """
    text_parts = []
    
    # --- Kline Data ---
    df_kline = data.get('kline', pd.DataFrame())
    if not df_kline.empty:
        text_parts.append("=== 4H KLINE DATA ===")
        text_parts.append(f"Total candles: {len(df_kline)}")
        text_parts.append(f"Time range: {df_kline['close_time'].min()} to {df_kline['close_time'].max()}")
        text_parts.append("\nPrice action:")
        
        for _, row in df_kline.iterrows():
            text_parts.append(
                f"  {row['close_time']}: O={row['open_price']:.2f}, H={row['high_price']:.2f}, "
                f"L={row['low_price']:.2f}, C={row['close_price']:.2f}, V={row['volume']:.2f}"
            )
        
        text_parts.append(f"\nPrice Summary:")
        text_parts.append(f"  Range: {df_kline['low_price'].min():.2f} - {df_kline['high_price'].max():.2f}")
        text_parts.append(f"  Avg Volume: {df_kline['volume'].mean():.2f}")
        text_parts.append(f"  Change: {((df_kline['close_price'].iloc[-1] / df_kline['close_price'].iloc[0] - 1) * 100):.2f}%")
    else:
        text_parts.append("=== 4H KLINE DATA ===")
        text_parts.append("No kline data available")
    
    text_parts.append("\n")
    
    # --- Funding Rate Data ---
    funding = data.get('funding', {})
    if funding:
        text_parts.append("=== FUNDING RATE DATA ===")
        text_parts.append(f"Total Records: {funding['count']}")

        series = funding.get('series', [])
        if series and len(series) > 0:
            # Extract rates for better analysis
            rates = [point['rate'] for point in series]
            start_rate = rates[0]
            end_rate = rates[-1]
            avg_rate = sum(rates) / len(rates)
            min_rate = min(rates)
            max_rate = max(rates)

            text_parts.append("\nFunding Rate Summary:")
            text_parts.append(f"  Start Rate: {start_rate:.6f} ({start_rate*100:.4f}%)")
            text_parts.append(f"  End Rate: {end_rate:.6f} ({end_rate*100:.4f}%)")
            text_parts.append(f"  Average Rate: {avg_rate:.6f} ({avg_rate*100:.4f}%)")
            text_parts.append(f"  Range: {min_rate:.6f} to {max_rate:.6f}")

            # Determine trend
            rate_change = ((end_rate - start_rate) / abs(start_rate) * 100) if start_rate != 0 else 0
            trend = "Increasing" if rate_change > 10 else "Decreasing" if rate_change < -10 else "Stable"
            text_parts.append(f"  Trend: {trend} ({rate_change:+.1f}% change)")

            # Show sampled funding rates (evenly distributed)
            text_parts.append(f"\nFunding Rate Timeline (showing {len(series)} key points):")
            for point in series:
                text_parts.append(f"  {point['ts']}: {point['rate']:.6f} ({point['rate']*100:.4f}%)")
        else:
            text_parts.append(f"Base Rate: {funding.get('base_rate', 0):.6f}")
    else:
        text_parts.append("=== FUNDING RATE DATA ===")
        text_parts.append("No funding rate data available")
    
    text_parts.append("\n")
    
    # --- Open Interest Data ---
    df_oi = data.get('open_interest', pd.DataFrame())
    if not df_oi.empty:
        text_parts.append("=== OPEN INTEREST DATA (4H) ===")
        text_parts.append(f"Total Records: {len(df_oi)}")
        text_parts.append(f"Time range: {df_oi['ts'].min()} to {df_oi['ts'].max()}")

        # Calculate OI statistics
        start_oi_usd = df_oi['open_interest_usd'].iloc[0]
        end_oi_usd = df_oi['open_interest_usd'].iloc[-1]
        avg_oi_usd = df_oi['open_interest_usd'].mean()
        max_oi_usd = df_oi['open_interest_usd'].max()
        min_oi_usd = df_oi['open_interest_usd'].min()

        start_oi_coin = df_oi['open_interest_coin'].iloc[0]
        end_oi_coin = df_oi['open_interest_coin'].iloc[-1]

        oi_change_pct = ((end_oi_usd - start_oi_usd) / start_oi_usd * 100)
        oi_coin_change_pct = ((end_oi_coin - start_oi_coin) / start_oi_coin * 100)

        text_parts.append("\nOI Summary (USD):")
        text_parts.append(f"  Start: ${start_oi_usd:,.0f}")
        text_parts.append(f"  End: ${end_oi_usd:,.0f}")
        text_parts.append(f"  Change: {oi_change_pct:+.2f}%")
        text_parts.append(f"  Average: ${avg_oi_usd:,.0f}")
        text_parts.append(f"  Range: ${min_oi_usd:,.0f} - ${max_oi_usd:,.0f}")

        # Determine OI trend
        if oi_change_pct > 5:
            trend = "Increasing (bullish sentiment)"
        elif oi_change_pct < -5:
            trend = "Decreasing (bearish sentiment)"
        else:
            trend = "Stable (neutral)"
        text_parts.append(f"  Trend: {trend}")

        text_parts.append(f"\nOI Summary (Coin):")
        text_parts.append(f"  Start: {start_oi_coin:,.2f}")
        text_parts.append(f"  End: {end_oi_coin:,.2f}")
        text_parts.append(f"  Change: {oi_coin_change_pct:+.2f}%")

        # Sample OI data (show first, middle, and last few records)
        sample_size = min(8, len(df_oi))
        if sample_size < len(df_oi):
            # Show start, middle, and end
            indices = [0, 1, 2] + [len(df_oi)//2 - 1, len(df_oi)//2, len(df_oi)//2 + 1] + [len(df_oi) - 3, len(df_oi) - 2, len(df_oi) - 1]
            indices = sorted(list(set([i for i in indices if 0 <= i < len(df_oi)])))
        else:
            indices = range(len(df_oi))

        text_parts.append(f"\nOI Timeline (showing {len(indices)} key points):")
        for idx in indices:
            row = df_oi.iloc[idx]
            text_parts.append(
                f"  {row['ts']}: ${row['open_interest_usd']:,.0f} ({row['open_interest_coin']:,.1f} coins)"
            )
    else:
        text_parts.append("=== OPEN INTEREST DATA ===")
        text_parts.append("No open interest data available")
    
    return "\n".join(text_parts)


def prepare_simulator_data_text(df: pd.DataFrame) -> str:
    """
    Convert simulator DataFrame to formatted text
    
    Args:
        df: DataFrame from fetch_simulator_data() with close_time and price
    
    Returns:
        Formatted text string
    """
    if df.empty:
        return "No simulator data available"
    
    text_parts = []
    text_parts.append(f"=== 1H PRICE DATA ===")
    text_parts.append(f"Total candles: {len(df)}")
    text_parts.append(f"Time range: {df['close_time'].min()} to {df['close_time'].max()}")
    text_parts.append("\nPrice series:")
    
    for _, row in df.iterrows():
        text_parts.append(f"  {row['close_time']}: {row['price']:.2f}")
    
    text_parts.append(f"\nPrice Summary:")
    text_parts.append(f"  Min: {df['price'].min():.2f}")
    text_parts.append(f"  Max: {df['price'].max():.2f}")
    text_parts.append(f"  Avg: {df['price'].mean():.2f}")
    text_parts.append(f"  Change: {((df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100):.2f}%")
    
    return "\n".join(text_parts)


def create_batch_jsonl(requests: List[Dict], filepath: str) -> str:
    """
    Create JSONL file for OpenAI Batch API
    
    Args:
        requests: List of request objects (must be JSON serializable)
        filepath: Output file path
    
    Returns:
        The file path
    """
    def to_serializable(obj):
        # Pydantic v2 models
        if isinstance(obj, BaseModel):
            return obj.model_dump()
        # Dataclasses
        if is_dataclass(obj):
            return asdict(obj)
        # Numpy numeric types
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        # Numpy arrays
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        # Generic objects
        if hasattr(obj, "__dict__"):
            return vars(obj)
        # Fallback
        return str(obj)

    with open(filepath, "w", encoding="utf-8") as f:
        for req in requests:
            json_line = json.dumps(req, default=to_serializable, ensure_ascii=False)
            f.write(json_line + "\n")

    return filepath


def submit_batch(file_path: str, description: str, endpoint: str = "/v1/responses") -> str:
    """
    Submit batch job to OpenAI
    
    Args:
        file_path: Path to JSONL file
        description: Batch description
        endpoint: API endpoint (default: /v1/responses for Responses API)
    
    Returns:
        Batch ID
    """
    # Upload file
    with open(file_path, 'rb') as f:
        batch_file = client.files.create(
            file=f,
            purpose='batch'
        )
    
    # Create batch
    batch = client.batches.create(
        input_file_id=batch_file.id,
        endpoint=endpoint,
        completion_window="24h",
        metadata={"description": description}
    )
    
    return batch.id


def wait_for_batch(batch_id: str, check_interval: int = 60) -> Dict:
    """
    Wait for batch completion
    
    Args:
        batch_id: Batch ID
        check_interval: Seconds between checks
    
    Returns:
        Batch info dict
    """
    while True:
        batch = client.batches.retrieve(batch_id)
        
        print(f"Status: {batch.status} | Completed: {batch.request_counts.completed}/{batch.request_counts.total}")
        
        if batch.status == "completed":
            return batch
        elif batch.status in ["failed", "expired", "cancelled"]:
            raise Exception(f"Batch failed with status: {batch.status}")
        
        time.sleep(check_interval)


def download_batch_results(batch_info: Dict, output_path: str) -> List[Dict]:
    """
    Download and parse batch results
    
    Args:
        batch_info: Batch info from retrieve
        output_path: Where to save results JSONL
    
    Returns:
        List of result objects
    """
    # Download output file
    output_file = client.files.content(batch_info.output_file_id)
    
    # Save to file
    with open(output_path, 'wb') as f:
        f.write(output_file.read())
    
    # Parse results
    results = []
    with open(output_path, 'r') as f:
        for line in f:
            results.append(json.loads(line))
    
    return results


print("✓ Batch helper functions defined")

✓ Batch helper functions defined


## Phase 1: Technical Analysis (Batch with Responses API)

In [8]:
def create_ta_batch_requests(
    windows: List[BacktestWindow],
    symbol: str = "BTCUSDT"
) -> List[Dict]:
    """
    Create batch requests for Technical Analysis phase using Responses API format
    
    Uses fetch_market_data() to get:
    - 4h klines (OHLCV)
    - Compressed funding rates
    - Open interest (4h aggregated)
    
    Args:
        windows: List of backtest windows
        symbol: Trading symbol
    
    Returns:
        List of batch request objects in Responses API format
    """
    requests = []
    
    for window in windows:
        # Fetch comprehensive market data for this window
        market_data = fetch_market_data(symbol, window.start_date, window.end_date)
        market_data_text = prepare_market_data_text(market_data)
        
        # Create prompt
        prompt = TA_AGENT_PROMPT.format(
            symbol=symbol,
            start_date=window.start_date.isoformat(),
            end_date=window.end_date.isoformat(),
            market_data=market_data_text
        )
        
        # Create batch request using Responses API format
        request = {
            "custom_id": f"ta_window_{window.window_id}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-5-mini",
                "input": [  # Using 'input' instead of 'messages'
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": "Phân tích kỹ thuật đồng theo xu hướng ngắn hạn trong 24-72 giờ tới dựa trên dữ liệu từ thị trường"}
                ]
            }
        }
        
        requests.append(request)
    
    return requests


def process_ta_results(
    results: List[Dict],
    windows: List[BacktestWindow]
) -> List[BacktestWindow]:
    """
    Process TA batch results and update windows
    
    Args:
        results: Batch results from OpenAI
        windows: Original windows
    
    Returns:
        Updated windows with analysis_data
    """
    # Create lookup map
    results_map = {}
    for result in results:
        custom_id = result['custom_id']
        window_id = int(custom_id.split('_')[-1])
        
        if result['response']['status_code'] == 200:
            # Extract text from Responses API output
            # The output array contains objects with different types:
            # - Index 0: reasoning block (type: "reasoning")
            # - Index 1+: message blocks (type: "message")
            # We need to find the message block and extract text from it
            
            output = result['response']['body']['output']
            content = None
            
            # Iterate through output to find the message type
            for item in output:
                if item.get('type') == 'message':
                    # Extract text from the message's content array
                    if 'content' in item and len(item['content']) > 0:
                        for content_item in item['content']:
                            if content_item.get('type') == 'output_text':
                                content = content_item.get('text')
                                break
                    break
            
            if content:
                results_map[window_id] = content
    
    # Update windows
    for window in windows:
        if window.window_id in results_map:
            window.analysis_data = results_map[window.window_id]
    
    return windows


print("✓ Phase 1 (TA) functions defined")

✓ Phase 1 (TA) functions defined


## Phase 2: Trade Simulation (Batch with Responses API + Structured Output)

In [9]:
def create_simulator_batch_requests(
    windows: List[BacktestWindow],
    symbol: str = "BTCUSDT",
    future_days: int = 7
) -> List[Dict]:
    """
    Create batch requests for Simulator phase using Responses API with text_format
    
    Uses fetch_simulator_data() to get:
    - 1h klines (price + time only)
    
    Args:
        windows: Windows with TA analysis
        symbol: Trading symbol
        future_days: Days of future data to simulate on
    
    Returns:
        List of batch request objects in Responses API format with structured output
    """
    requests = []
    
    
    for window in windows:
        if not window.analysis_data:
            continue  # Skip if no TA analysis
        
        # Fetch future market data (for simulation) - 1h data
        future_start = window.end_date
        future_end = window.end_date + timedelta(days=future_days)
        
        df_future = fetch_simulator_data(symbol, future_start, future_end)
        future_data_text = prepare_simulator_data_text(df_future)
        
        # Create prompt
        prompt = SIMULATOR_AGENT_PROMPT.format(
            ta_analysis=window.analysis_data,
            future_market_data=future_data_text,
            trade_record_schema="See text_format parameter"
        )
        
        # Create batch request using Responses API format with text_format
        request = {
            "custom_id": f"sim_window_{window.window_id}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-5-mini",
                "input": [  # Using 'input' instead of 'messages'
                    {"role": "system", "content": "You are a professional trading simulator."},
                    {"role": "user", "content": prompt}
                ],
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "TradeSimulationOutput",
                    "schema": {
                        "title": "Kết quả mô phỏng giao dịch",
                        "description": "Cấu trúc đầu ra của mô phỏng giao dịch, gồm danh sách chi tiết các lệnh được thực hiện.",
                        "type": "object",
                        "properties": {
                            "trades": {
                                "type": "array",
                                "description": "Danh sách các giao dịch được mô phỏng.",
                                "items": {
                                    "type": "object",
                                    "description": "Chi tiết từng giao dịch trong mô phỏng.",
                                    "properties": {
                                        "id": {
                                            "type": "integer",
                                            "description": "Số thứ tự của giao dịch."
                                        },
                                        "entry_time": {
                                            "type": "string",
                                            "format": "date-time",
                                            "description": "Thời gian vào lệnh (UTC+7)."
                                        },
                                        "exit_time": {
                                            "type": "string",
                                            "format": "date-time",
                                            "description": "Thời gian thoát lệnh (UTC+7)."
                                        },
                                        "order_type": {
                                            "type": "string",
                                            "description": "Loại lệnh: Long hoặc Short."
                                        },
                                        "entry_price": {
                                            "type": "number",
                                            "description": "Giá vào lệnh theo phân tích kỹ thuật."
                                        },
                                        "exit_price": {
                                            "type": "number",
                                            "description": "Giá thoát lệnh thực tế."
                                        },
                                        "target_price": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Giá mục tiêu (TP) nếu có."
                                        },
                                        "stop_price": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Giá dừng lỗ (SL) nếu có."
                                        },
                                        "pnl_expected": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Lợi nhuận dự kiến (%) theo TA."
                                        },
                                        "pnl_real": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Lợi nhuận hoặc lỗ thực tế (%)."
                                        },
                                        "deviation": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Sai lệch giữa dự kiến và thực tế (%)."
                                        },
                                        "holding_time_hours": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Thời gian giữ lệnh (giờ)."
                                        },
                                        "technical_reason": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Lý do kỹ thuật mở lệnh."
                                        },
                                        "ta_reference": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Tham chiếu tới báo cáo phân tích kỹ thuật."
                                        },
                                        "result": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Kết quả: Lãi hoặc Lỗ."
                                        },
                                        "ta_assessment": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Nhận định của TA về kết quả."
                                        },
                                        "market_result": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Kết quả thực tế của thị trường."
                                        },
                                        "deviation_reason": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Nguyên nhân chênh lệch giữa TA và thực tế."
                                        },
                                        "improvement_note": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Gợi ý cải thiện chiến lược."
                                        },
                                        "volume_condition": {
                                            "anyOf": [{"type": "string"}, {"type": "null"}],
                                            "description": "Tình trạng volume tại thời điểm giao dịch."
                                        },
                                        "oi_change_24h": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Thay đổi OI trong 24h (%)."
                                        },
                                        "funding_rate": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Funding rate tại thời điểm vào lệnh."
                                        },
                                        "atr_value": {
                                            "anyOf": [{"type": "number"}, {"type": "null"}],
                                            "description": "Giá trị ATR tại thời điểm vào lệnh."
                                        }
                                    },
                                    "required": [
                                        "id",
                                        "entry_time",
                                        "exit_time",
                                        "order_type",
                                        "entry_price",
                                        "exit_price",
                                        "target_price",
                                        "stop_price",
                                        "pnl_expected",
                                        "pnl_real",
                                        "deviation",
                                        "holding_time_hours",
                                        "technical_reason",
                                        "ta_reference",
                                        "result",
                                        "ta_assessment",
                                        "market_result",
                                        "deviation_reason",
                                        "improvement_note",
                                        "volume_condition",
                                        "oi_change_24h",
                                        "funding_rate",
                                        "atr_value"
                                    ],
                                    "additionalProperties": False
                                }
                            }
                        },
                        "required": ["trades"],
                        "additionalProperties": False
                    },
                    "strict": True
                }
            }
        }
    }

        requests.append(request)
    
    return requests


def process_simulator_results(results: List[Dict], windows: List["BacktestWindow"]) -> List["BacktestWindow"]:
    """
    Process Simulator batch results and update windows
    """
    results_map = {}

    for result in results:
        custom_id = result.get("custom_id")
        window_id = int(custom_id.split("_")[-1])

        try:
            response = result["response"]
            if response.get("status_code") != 200:
                print(f"⚠️ Window {window_id}: Request failed ({response.get('status_code')})")
                results_map[window_id] = []
                continue

            body = response["body"]
            output_parsed = None

            # Responses API structured output text
            if "output" in body and isinstance(body["output"], list):
                for item in body["output"]:
                    if item.get("type") == "message":
                        for content in item.get("content", []):
                            if content.get("type") == "output_text" and "text" in content:
                                try:
                                    output_parsed = json.loads(content["text"])
                                except json.JSONDecodeError as e:
                                    print(f"❌ JSON parse error for window {window_id}: {e}")
                                break

            if output_parsed:
                trades = output_parsed.get("trades", [])
                results_map[window_id] = trades
            else:
                print(f"⚠️ No valid structured output for window {window_id}")
                results_map[window_id] = []

        except Exception as e:
            print(f"❌ Error parsing window {window_id}: {e}")
            import traceback
            traceback.print_exc()
            results_map[window_id] = []

    # Update windows
    for window in windows:
        if window.window_id in results_map:
            window.trades = results_map[window.window_id]

    return windows


print("✓ Phase 2 (Simulator) functions defined")

✓ Phase 2 (Simulator) functions defined


## Phase 3: Review & Analysis (Batch with Responses API + Code Interpreter)

In [10]:
def _get(trade, key, default=None):
    # Hỗ trợ cả dict và object
    if isinstance(trade, dict):
        return trade.get(key, default)
    return getattr(trade, key, default)

def create_reviewer_batch_requests(
    windows: List[BacktestWindow],
    symbol: str = "BTCUSDT",
    review_days: int = 7
) -> List[Dict]:
    requests = []

    for window in windows:
        if not getattr(window, "analysis_data", None):
            continue

        review_start = window.end_date
        review_end = window.end_date + timedelta(days=review_days)

        actual_data = fetch_market_data(symbol, review_start, review_end)
        actual_data_text = prepare_market_data_text(actual_data)

        # --- Simulation summary (fix truy cập dict) ---
        if getattr(window, "trades", None) and len(window.trades) > 0:
            trades = window.trades

            # Tính tổng/đếm an toàn
            def _to_num(x):
                try:
                    return float(x)
                except (TypeError, ValueError):
                    return None

            total_trades = len(trades)
            pnl_vals = [_to_num(_get(t, "pnl_real")) for t in trades]
            pnl_clean = [v for v in pnl_vals if v is not None]
            total_pnl = sum(pnl_clean) if pnl_clean else 0.0
            winning_trades = sum(1 for v in pnl_clean if v > 0)

            simulation_summary = (
                "=== SIMULATION SUMMARY ===\n"
                f"Total Trades: {total_trades}\n"
                f"Winning Trades: {winning_trades} ({(winning_trades/total_trades*100):.1f}%)\n"
                f"Total P&L: {total_pnl:.2f}%\n"
                f"Average P&L per trade: {(total_pnl/total_trades):.2f}%\n\n"
                "Trade Details:\n"
            )

            for t in trades:
                tid = _get(t, "id", "?")
                otype = _get(t, "order_type", "N/A")
                eprice = _get(t, "entry_price", "N/A")
                xprice = _get(t, "exit_price", "N/A")
                etime = _get(t, "entry_time", "N/A")
                xtime = _get(t, "exit_time", "N/A")
                pnl_real = _get(t, "pnl_real")
                pnl_expected = _get(t, "pnl_expected")
                reason = _get(t, "technical_reason", "")

                def fmt_pct(v):
                    return f"{float(v):.2f}%" if isinstance(v, (int, float)) or (isinstance(v, str) and v.replace('.', '', 1).isdigit()) else "N/A"

                simulation_summary += (
                    f"\n- Trade #{tid}: {otype}\n"
                    f"  Entry: {eprice} at {etime}\n"
                    f"  Exit: {xprice} at {xtime}\n"
                    f"  P&L: {fmt_pct(pnl_real)} (Expected: {fmt_pct(pnl_expected)})\n"
                    f"  Reason: {reason}\n"
                )
        else:
            simulation_summary = "No trades were executed based on TA analysis."

        # Prompt cho Reviewer
        prompt = TA_REVIEWER_AGENT_PROMPT.format(
            ta_analysis=window.analysis_data,
            actual_market_data=actual_data_text,
            simulation_summary=simulation_summary
        )

        # Yêu cầu batch theo Responses API (có code_interpreter + schema)
        request = {
            "custom_id": f"review_window_{window.window_id}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-5-mini",
                "input": [
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": "Hãy tiến hành đánh giá độ chính xác của phân tích kỹ thuật. Sử dụng code interpreter để tính toán và phân tích."}
                ],
                "tools": [
                    {"type": "code_interpreter", "container": {"type": "auto"}}
                ],
               
            }}
        requests.append(request)
        window.actual_market_data = actual_data_text

    return requests



def process_reviewer_results(results: List[Dict], windows: List[BacktestWindow]) -> List[BacktestWindow]:
    """
    Process TA Reviewer batch results and update windows
    (For plain-text LLM outputs)
    """
    results_map = {}
    
    for result in results:
        custom_id = result['custom_id']
        window_id = int(custom_id.split('_')[-1])
        
        if result['response']['status_code'] == 200:
            try:
                body = result['response']['body']
                review_text = None

                # Try extracting plain text output
                if 'output' in body and len(body['output']) > 0:
                    messages = [item for item in body['output'] if item.get('type') == 'message']
                    if messages:
                        last_message = messages[-1]
                        for content_item in last_message.get('content', []):
                            if content_item.get('type') == 'output_text':
                                review_text = content_item['text']
                                break

                if not review_text:
                    print(f"Warning: No text output found for window {window_id}")
                
                results_map[window_id] = review_text

            except Exception as e:
                print(f"Error parsing review for window {window_id}: {e}")
                import traceback
                traceback.print_exc()
                results_map[window_id] = None
    
    # Update windows
    for window in windows:
        if window.window_id in results_map:
            window.review = results_map[window.window_id]
    
    return windows



print("✓ Phase 3 (TA Reviewer) functions defined")

✓ Phase 3 (TA Reviewer) functions defined


## Main Workflow Orchestration

In [ ]:
def safe_download_with_fallback(batch_info, output_path, step_name):
    """Try to download batch results safely, handle missing file_id gracefully."""
    try:
        if not batch_info or not getattr(batch_info, "output_file_id", None):
            print(f"[{step_name}] output_file_id is None — batch may have failed or been flagged.")
            print(f"    → Creating empty fallback file: {output_path}")
            with open(output_path, "w") as f:
                f.write("[]")
            return []

        results = download_batch_results(batch_info, output_path)
        return results

    except Exception as e:
        print(f"❌ [{step_name}] Failed to download batch results: {e}")
        print(f"    → Creating empty fallback file: {output_path}")
        with open(output_path, "w") as f:
            f.write("[]")
        return []

def run_3_agent_backtest(
    start_date: datetime,
    end_date: datetime,
    symbol: str = "eth",
    window_size_days: int = 7,
    slide_days: int = 3,
    output_dir: str = "./backtest_results"
) -> List["BacktestWindow"]:
    """
    Run complete 3-agent backtesting workflow using Batch API with Responses API format
    
    Workflow:
    1. TA Agent: Technical Analysis
    2. Simulator Agent: Trade Simulation
    3. Reviewer Agent: TA Review
    (Total: 7 steps end-to-end)
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("3-AGENT BACKTESTING WORKFLOW (Batch API + Responses API Format)")
    print("=" * 80)
    print(f"Symbol: {symbol}")
    print(f"Period: {start_date} → {end_date}")
    print(f"Window: {window_size_days} days, Slide: {slide_days} days\n")

    # ---------- STEP 1 ----------
    print("[1/7] Generating sliding windows...")
    windows = generate_sliding_windows(start_date, end_date, window_size_days, slide_days)
    print(f"      ✓ Generated {len(windows)} windows\n")

    # ---------- STEP 2 ----------
    print("[2/7] Creating Technical Analysis (TA) batch requests...")
    ta_requests = create_ta_batch_requests(windows, symbol)
    ta_file = create_batch_jsonl(ta_requests, f"{output_dir}/batch_ta_requests.jsonl")
    print(f"      ✓ Created {len(ta_requests)} TA requests\n")

    # ---------- STEP 3 ----------
    print("[3/7] Submitting TA batch to OpenAI...")
    ta_batch_id = submit_batch(ta_file, "Technical Analysis Batch", endpoint="/v1/responses")
    print(f"      Batch ID: {ta_batch_id}")
    print("      Waiting for completion...")
    ta_batch_info = wait_for_batch(ta_batch_id)
    ta_results = safe_download_with_fallback(ta_batch_info, f"{output_dir}/batch_ta_results.jsonl", "TA Batch")
    windows = process_ta_results(ta_results, windows)
    print(f"      ✓ TA analysis completed\n")

    # ---------- STEP 4 ----------
    print("[4/7] Creating Trade Simulation batch requests...")
    sim_requests = create_simulator_batch_requests(windows, symbol)
    sim_file = create_batch_jsonl(sim_requests, f"{output_dir}/batch_sim_requests.jsonl")
    print(f"      ✓ Created {len(sim_requests)} simulator requests\n")

    # ---------- STEP 5 ----------
    print("[5/7] Submitting Simulator batch to OpenAI...")
    sim_batch_id = submit_batch(sim_file, "Trade Simulation Batch", endpoint="/v1/responses")
    print(f"      Batch ID: {sim_batch_id}")
    print("      Waiting for completion...")
    sim_batch_info = wait_for_batch(sim_batch_id)
    sim_results = safe_download_with_fallback(sim_batch_info, f"{output_dir}/batch_sim_results.jsonl", "Simulator Batch")
    windows = process_simulator_results(sim_results, windows)
    print(f"      ✓ Trade simulation completed\n")

    # ---------- STEP 6 ----------
    print("[6/7] Creating TA Review batch requests...")
    review_requests = create_reviewer_batch_requests(windows, symbol)
    review_file = create_batch_jsonl(review_requests, f"{output_dir}/batch_review_requests.jsonl")
    print(f"      ✓ Created {len(review_requests)} review requests\n")

    # ---------- STEP 7 ----------
    print("[7/7] Submitting Reviewer batch to OpenAI...")
    review_batch_id = submit_batch(review_file, "TA Review Batch", endpoint="/v1/responses")
    print(f"      Batch ID: {review_batch_id}")
    print("      Waiting for completion...")
    review_batch_info = wait_for_batch(review_batch_id)
    review_results = safe_download_with_fallback(review_batch_info, f"{output_dir}/batch_review_results.jsonl", "Reviewer Batch")
    windows = process_reviewer_results(review_results, windows)
    print(f"      ✓ TA review & analysis completed\n")

    # ---------- SAVE FINAL ----------
    final_output = f"{output_dir}/backtest_final_results.json"
    with open(final_output, "w") as f:
        json.dump([w.model_dump() for w in windows], f, indent=2, ensure_ascii=False, default=str)

    total_trades = sum(len(w.trades) for w in windows if w.trades)
    total_reviews = sum(1 for w in windows if getattr(w, "review", None))

    print("=" * 80)
    print("BACKTEST COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Symbol: {symbol}")
    print(f"Windows processed: {len(windows)}")
    print(f"Total trades simulated: {total_trades}")
    print(f"TA predictions reviewed: {total_reviews}")
    print(f"Results saved to: {final_output}\n")

    return windows



## Manual Testing: Single Window (7 Steps)

Run these cells sequentially to test each step of the workflow manually with a single window.

In [74]:
# STEP 1: Generate Single Window
# Test period: 2025-10-01 → 2025-10-08 (7 days)

TEST_SYMBOL = "eth"
TEST_START = datetime(2025, 10, 1, tzinfo=pytz.UTC)
TEST_END = datetime(2025, 10, 8, tzinfo=pytz.UTC)
OUTPUT_DIR = "./test_single_window"

# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Generate single window
test_windows = [BacktestWindow(
    window_id=0,
    start_date=TEST_START,
    end_date=TEST_END
)]

print("=" * 80)
print("STEP 1: Generate Single Window")
print("=" * 80)
print(f"Symbol: {TEST_SYMBOL}")
print(f"Window: {test_windows[0].start_date} → {test_windows[0].end_date}")
print(f"Duration: {(TEST_END - TEST_START).days} days")
print(f"✓ Window created successfully\n")

STEP 1: Generate Single Window
Symbol: eth
Window: 2025-10-01 00:00:00+00:00 → 2025-10-08 00:00:00+00:00
Duration: 7 days
✓ Window created successfully



In [75]:
# STEP 2: Create TA Batch Requests

print("=" * 80)
print("STEP 2: Create Technical Analysis Batch Requests")
print("=" * 80)

# Create TA batch requests
ta_requests = create_ta_batch_requests(test_windows, TEST_SYMBOL)
ta_file = create_batch_jsonl(ta_requests, f"{OUTPUT_DIR}/batch_ta_requests.jsonl")

print(f"✓ Created {len(ta_requests)} TA requests")
print(f"✓ Saved to: {ta_file}")
print(f"\nRequest preview:")
print(f"  custom_id: {ta_requests[0]['custom_id']}")
print(f"  model: {ta_requests[0]['body']['model']}")
print(f"  input messages: {len(ta_requests[0]['body']['input'])}")
print()

STEP 2: Create Technical Analysis Batch Requests
✓ Created 1 TA requests
✓ Saved to: ./test_single_window/batch_ta_requests.jsonl

Request preview:
  custom_id: ta_window_0
  model: gpt-5-mini
  input messages: 2



In [19]:
# STEP 3: Submit and Process TA Batch

print("=" * 80)
print("STEP 3: Submit TA Batch to OpenAI")
print("=" * 80)

# Submit batch
ta_batch_id = submit_batch(ta_file, "Test TA Batch - Single Window", endpoint="/v1/responses")
print(f"✓ Batch submitted")
print(f"  Batch ID: {ta_batch_id}")
print(f"\nWaiting for completion (checking every 30 seconds)...")

# Wait for completion
ta_batch_info = wait_for_batch(ta_batch_id, check_interval=30)

# Download and process results
ta_results = download_batch_results(ta_batch_info, f"{OUTPUT_DIR}/batch_ta_results.jsonl")


STEP 3: Submit TA Batch to OpenAI
✓ Batch submitted
  Batch ID: batch_690c1eba75d481908ec16db0b3fedf72

Waiting for completion (checking every 30 seconds)...
Status: validating | Completed: 0/0
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: completed | Completed: 1/1


In [20]:
test_windows = process_ta_results(ta_results, test_windows)

print(f"\n✓ TA batch completed")
print(f"  Status: {ta_batch_info.status}")
print(f"  Results: {len(ta_results)} responses")
print(f"  Windows updated: {sum(1 for w in test_windows if w.analysis_data)}")

# Display TA analysis preview
if test_windows[0].analysis_data:
    print(f"\nTA Analysis Preview (first 500 chars):")
    print("-" * 80)
    print(test_windows[0].analysis_data[:500] + "...")
    print("-" * 80)
print()


✓ TA batch completed
  Status: completed
  Results: 1 responses
  Windows updated: 1

TA Analysis Preview (first 500 chars):
--------------------------------------------------------------------------------
Dưới đây là phân tích kỹ thuật ngắn hạn (khung 4H) cho ETH dựa trên dữ liệu 4H kết thúc tại 2025-10-07 23:00 (UTC+7). Tôi đã dùng dữ liệu giá (O/H/L/C), volume, OI và funding như bạn cung cấp.

Tóm tắt nhanh
- Giá hiện tại (4H close cuối): 4,511.99 USD
- Giá mở cửa của nến 4H gần nhất: 4,708.82 USD
- Biên độ 24h: High = 4,755.00 USD, Low = 4,477.43 USD
- Thay đổi 24h (so với close 24h trước 2025-10-06 23:00 C=4,673.39): -3.46%
- Phạm vi dữ liệu 4H (khoảng 2025-10-01 → 2025-10-07): giá dao động 4...
--------------------------------------------------------------------------------



In [73]:
# STEP 4: Create Simulator Batch Requests

print("=" * 80)
print("STEP 4: Create Simulator Batch Requests")
print("=" * 80)

# Create Simulator batch requests
sim_requests = create_simulator_batch_requests(test_windows, TEST_SYMBOL, future_days=7)
sim_file = create_batch_jsonl(sim_requests, f"{OUTPUT_DIR}/batch_sim_requests.jsonl")

print(f"✓ Created {len(sim_requests)} Simulator requests")
print(f"✓ Saved to: {sim_file}")
print(f"\nRequest preview:")
print(f"  custom_id: {sim_requests[0]['custom_id']}")
print(f"  model: {sim_requests[0]['body']['model']}")
print(f"  has text_format: {'text_format' in sim_requests[0]['body']}")
print(f"  input messages: {len(sim_requests[0]['body']['input'])}")
print()

STEP 4: Create Simulator Batch Requests
✓ Created 0 Simulator requests
✓ Saved to: ./test_single_window/batch_sim_requests.jsonl

Request preview:


IndexError: list index out of range

In [36]:
# STEP 5: Submit and Process Simulator Batch

print("=" * 80)
print("STEP 5: Submit Simulator Batch to OpenAI")
print("=" * 80)

# Submit batch
sim_batch_id = submit_batch(sim_file, "Test Simulator Batch - Single Window", endpoint="/v1/responses")
print(f"✓ Batch submitted")
print(f"  Batch ID: {sim_batch_id}")
print(f"\nWaiting for completion (checking every 30 seconds)...")

# Wait for completion
sim_batch_info = wait_for_batch(sim_batch_id, check_interval=30)

# Download and process results
sim_results = download_batch_results(sim_batch_info, f"{OUTPUT_DIR}/batch_sim_results.jsonl")


STEP 5: Submit Simulator Batch to OpenAI
✓ Batch submitted
  Batch ID: batch_690c2a0774fc819099730681e218ca13

Waiting for completion (checking every 30 seconds)...
Status: validating | Completed: 0/0
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: completed | Completed: 1/1


In [41]:
test_windows = process_simulator_results(sim_results, test_windows)

print(f"\n✓ Simulator batch completed")
print(f"  Status: {sim_batch_info.status}")
print(f"  Results: {len(sim_results)} responses")
print(f"  Windows with trades: {sum(1 for w in test_windows if w.trades)}")

# Display trades summary
if test_windows[0].trades:
    print(f"\nTrades Summary:")
    print("-" * 80)
    print(f"  Total trades: {len(test_windows[0].trades)}")
    for i, trade in enumerate(test_windows[0].trades[:3]):  # Show first 3 trades
        # Truy cập theo key thay vì attribute
        order_type = trade.get("order_type", "N/A")
        entry = trade.get("entry_price", "N/A")
        exit_ = trade.get("exit_price", "N/A")
        pnl = trade.get("pnl_real", "N/A")
        print(f"  Trade {i+1}: {order_type} | Entry: {entry} | Exit: {exit_} | P&L: {pnl}%")
    if len(test_windows[0].trades) > 3:
        print(f"  ... and {len(test_windows[0].trades) - 3} more trades")
    print("-" * 80)
else:
    print(f"\n⚠ No trades generated")
print()



✓ Simulator batch completed
  Status: completed
  Results: 1 responses
  Windows with trades: 1

Trades Summary:
--------------------------------------------------------------------------------
  Total trades: 1
  Trade 1: No Trade (strategy constraint) | Entry: 0 | Exit: 0 | P&L: None%
--------------------------------------------------------------------------------



In [50]:
# STEP 6: Create TA Reviewer Batch Requests
print("=" * 80)
print("STEP 6: Create TA Reviewer Batch Requests")
print("=" * 80)

# Tạo batch request cho giai đoạn Reviewer
review_requests = create_reviewer_batch_requests(test_windows, TEST_SYMBOL, review_days=7)
review_file = create_batch_jsonl(review_requests, f"{OUTPUT_DIR}/batch_review_requests.jsonl")

# --- Thông tin tổng quan ---
print(f"✓ Created {len(review_requests)} Reviewer requests")
print(f"✓ Saved to: {review_file}")

# --- Hiển thị preview ---
preview = review_requests[0]["body"]
print("\nRequest preview:")
print(f"  custom_id: {review_requests[0]['custom_id']}")
print(f"  model: {preview['model']}")
print(f"  has text.format: {'text' in preview and 'format' in preview['text']}")
print(f"  has code_interpreter: {'tools' in preview}")
print(f"  input messages: {len(preview['input'])}")

# Nếu muốn xem sơ qua schema name
if 'text' in preview and 'format' in preview['text']:
    print(f"  schema name: {preview['text']['format'].get('name')}")
print()

STEP 6: Create TA Reviewer Batch Requests
✓ Created 1 Reviewer requests
✓ Saved to: ./test_single_window/batch_review_requests.jsonl

Request preview:
  custom_id: review_window_0
  model: gpt-5-mini
  has text.format: False
  has code_interpreter: True
  input messages: 2



In [51]:
# STEP 7: Submit and Process TA Reviewer Batch

print("=" * 80)
print("STEP 7: Submit TA Reviewer Batch to OpenAI")
print("=" * 80)

# Submit batch
review_batch_id = submit_batch(review_file, "Test TA Review Batch - Single Window", endpoint="/v1/responses")
print(f"✓ Batch submitted")
print(f"  Batch ID: {review_batch_id}")
print(f"\nWaiting for completion (checking every 30 seconds)...")

# Wait for completion
review_batch_info = wait_for_batch(review_batch_id, check_interval=30)

# Download and process results
review_results = download_batch_results(review_batch_info, f"{OUTPUT_DIR}/batch_review_results.jsonl")


STEP 7: Submit TA Reviewer Batch to OpenAI
✓ Batch submitted
  Batch ID: batch_690c46935d9081908bd917cbac2f3ad6

Waiting for completion (checking every 30 seconds)...
Status: validating | Completed: 0/0
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: in_progress | Completed: 0/1
Status: completed | Completed: 1/1


In [54]:
test_windows = process_reviewer_results(review_results, test_windows)

print(f"\n✓ TA Reviewer batch completed")
print(f"  Status: {review_batch_info.status}")
print(f"  Results: {len(review_results)} responses")
print(f"  Windows with reviews: {sum(1 for w in test_windows if w.review)}")

# Display review summary
if test_windows[0].review:
    print(f"\nTA Review Preview (first 800 chars):")
    print("-" * 80)
    print(test_windows[0].review[:800] + "...")
    print("-" * 80)
else:
    print(f"\n⚠ No review generated")

# Save final test results
final_output = f"{OUTPUT_DIR}/test_single_window_final.json"
with open(final_output, 'w') as f:
    json.dump(
        [w.model_dump() for w in test_windows],
        f,
        default=str,
        indent=2,
        ensure_ascii=False
    )

print(f"\n✓ Test completed successfully!")
print(f"  Final results saved to: {final_output}")
print("=" * 80)


✓ TA Reviewer batch completed
  Status: completed
  Results: 1 responses
  Windows with reviews: 1

TA Review Preview (first 800 chars):
--------------------------------------------------------------------------------
Tôi đã backtest và đánh giá bản phân tích kỹ thuật (TA) dựa trên dữ liệu thực tế bạn cung cấp. Tôi dùng code interpreter để tính toán các chỉ số, so sánh dự đoán với kết quả thực tế và vẽ biểu đồ (link tải hình ở cuối). Tóm tắt kết quả đánh giá bằng tiếng Việt như sau.

Tóm tắt nhanh (mốc thời gian)
- Điểm bắt đầu phân tích (TA): 2025-10-07 23:00 (close = 4,511.99 USD).
- Window 24h: đến 2025-10-08 23:00. Window 72h: đến 2025-10-10 23:00.
- Dữ liệu sử dụng: chuỗi 4H (42 nến) từ 2025-10-08 03:00 → 2025-10-14 23:00.

1) Kết quả tính toán (từ code)
- Max high trong 24h: 4,524.00 USD
- Min low trong 24h: 4,410.08 USD
- Max high trong 72h: 4,558.00 USD
- Min low trong 72h: 4,073.55 USD
- Max high toàn chuỗi: 4,558.00 USD; Min low toàn chuỗi: 3,435.00 USD
- Open Interest (OI):